# Natural-Language Data Preparation: Observation and Approval Tree

This notebook executes the read-only data checks, then pauses at each choice that could affect cleaning or features. You approve or disapprove each proposal in its own code cell. A disapproval pauses that branch; no alternative cleaning action is guessed.

## Natural-language request

> Load all bookings, split them using the project arrival-date periods, inspect the data, and show each proposed cleaning or feature action with its evidence. Wait for my approval or disapproval before treating any proposal as final. Do not inspect proof cancellation outcomes or train a model.

**Already requested and executed:** split by the contract periods. **Still awaiting your approval:** handling exact repeats, missing values, unusual values, candidate feature timing, log transforms, and hotel encoding.

Edit each `decision_...` cell to `"approve"` or `"disapprove"`, then run that cell and the final approval summary. `"pending"` makes no choice.

## 1. Load the raw file and supplied contract helpers

The raw table is kept unchanged. We add the arrival date to a separate working copy.

In [1]:
import sys
from pathlib import Path

# Let Python find the project helper files in src/.
# This works whether the notebook runs from the repo folder or from notebooks/.
repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

import numpy as np
import pandas as pd
import contract
import data

# Load the raw file. We never change raw_bookings.
raw_bookings = data.read_raw()

# Work on a copy and add one arrival_date column.
bookings = raw_bookings.copy()
bookings["arrival_date"] = data.arrival_date(bookings)

print("Rows:", len(raw_bookings))
print("Source columns:", len(raw_bookings.columns))
print("First arrival:", bookings["arrival_date"].min().date())
print("Last arrival:", bookings["arrival_date"].max().date())

Rows: 119390
Source columns: 32
First arrival: 2015-07-01
Last arrival: 2017-08-31


## 2. Split by arrival date

The contract function sets the boundaries. The summary below does not calculate cancellation rates.

In [2]:
# The contract decides the date boundaries for each period.
train, choose, proof = contract.split_periods(bookings)

# Show size, date range, and hotels for each period. No outcomes are read here.
periods = {"train": train, "choose": choose, "proof": proof}
for name, period in periods.items():
    print(name)
    print("  rows:", len(period))
    print("  arrivals:", period["arrival_date"].min().date(), "to", period["arrival_date"].max().date())
    print("  hotels:", sorted(period["hotel"].unique()))

print("Rows assigned:", len(train) + len(choose) + len(proof), "of", len(bookings))

train
  rows: 49157
  arrivals: 2015-07-01 to 2016-06-30
  hotels: ['City Hotel', 'Resort Hotel']
choose
  rows: 29546
  arrivals: 2016-07-01 to 2016-12-31
  hotels: ['City Hotel', 'Resort Hotel']
proof
  rows: 40687
  arrivals: 2017-01-01 to 2017-08-31
  hotels: ['City Hotel', 'Resort Hotel']
Rows assigned: 119390 of 119390


## 3. Natural-language request: profile training data quality

> On the training period only, show exact duplicate rows, columns with missing values, and simple range checks. Do not remove rows or fill values yet.

This keeps proof outcomes out of the EDA. Duplicate rows are not automatically errors because the file has no reservation ID to distinguish a repeated record from two identical bookings.

In [3]:
print("Training rows:", len(train))
print("Training columns:", len(train.columns))

# duplicated() marks a row True if an identical row appeared earlier.
print("Exact repeated rows after first occurrence:", train.duplicated().sum())

# Count missing values per column, then keep only columns that have some.
missing_counts = train.isna().sum()
missing_counts = missing_counts[missing_counts > 0]
missing_counts = missing_counts.sort_values(ascending=False)
print("\nMissing values in training data:")
print(missing_counts.to_string())

# Simple range check: smallest value, largest value, and how many are negative.
range_columns = [
    "adults", "children", "babies",
    "stays_in_weekend_nights", "stays_in_week_nights",
    "adr", "previous_cancellations", "previous_bookings_not_canceled",
]
range_summary = pd.DataFrame({
    "min": train[range_columns].min(),
    "max": train[range_columns].max(),
    "negative_values": (train[range_columns] < 0).sum(),
})
print("\nRange checks (training period):")
print(range_summary.to_string())

Training rows: 49157
Training columns: 33
Exact repeated rows after first occurrence: 16291

Missing values in training data:
company     46123
agent        6960
country       284
children        4

Range checks (training period):
                                min     max  negative_values
adults                          0.0    55.0                0
children                        0.0    10.0                0
babies                          0.0    10.0                0
stays_in_weekend_nights         0.0    18.0                0
stays_in_week_nights            0.0    42.0                0
adr                             0.0  5400.0                0
previous_cancellations          0.0    26.0                0
previous_bookings_not_canceled  0.0    36.0                0


### 3a. Profile all source columns on train

This compact profile shows type, missingness, distinct values, and the three most common values for every source column. The numeric table adds min, median, 99th percentile, max, and negative-value count. All statistics here use train only.

In [4]:
# Use only the original source columns (leave out the added arrival_date).
train_source = train[raw_bookings.columns]

# One row per column: its type, missing count, and number of distinct values.
column_profile = pd.DataFrame({
    "type": train_source.dtypes.astype(str),
    "missing": train_source.isna().sum(),
    "distinct": train_source.nunique(),
})

# Add the three most common values in each column (missing values included).
top_values = []
for column in train_source.columns:
    counts = train_source[column].value_counts(dropna=False).head(3)
    top_values.append(str(counts.to_dict()))
column_profile["top_values"] = top_values

print(column_profile.to_string())

# Extra summary for the numeric columns only.
numeric_data = train_source.select_dtypes(include="number")
numeric_profile = pd.DataFrame({
    "min": numeric_data.min(),
    "median": numeric_data.median(),
    "p99": numeric_data.quantile(0.99),
    "max": numeric_data.max(),
    "negative_values": (numeric_data < 0).sum(),
})
print("\nNumeric source-column summary:")
print(numeric_profile.to_string())

                                   type  missing  distinct                                                        top_values
hotel                               str        0         2                      {'City Hotel': 31623, 'Resort Hotel': 17534}
is_canceled                       int64        0         2                                              {0: 31572, 1: 17585}
lead_time                         int64        0       379                                       {0: 3338, 1: 1764, 2: 1021}
arrival_date_year                 int64        0         2                                        {2016: 27161, 2015: 21996}
arrival_date_month                  str        0        12                        {'May': 5478, 'April': 5428, 'June': 5292}
arrival_date_week_number          int64        0        53                                    {25: 1468, 21: 1452, 18: 1437}
arrival_date_day_of_month         int64        0        31                                     {17: 2147, 5: 1975, 19: 1852}


### 3b. Check cancellation patterns by training month

This checks whether the target varies over the training timeline and by hotel. It does not use choose or proof outcomes.

In [5]:
# Turn each arrival date into its month, e.g. 2015-07.
arrival_month = train["arrival_date"].dt.to_period("M")

# For each month and hotel: number of bookings and share cancelled.
train_monthly = train.groupby([arrival_month, "hotel"])["is_canceled"].agg(
    bookings="count",
    cancellation_rate="mean",
)
train_monthly["cancellation_rate"] = (train_monthly["cancellation_rate"] * 100).round(1)
train_monthly = train_monthly.rename(columns={"cancellation_rate": "cancellation_rate_%"})
print(train_monthly.to_string())

                           bookings  cancellation_rate_%
arrival_date hotel                                      
2015-07      City Hotel        1398                 67.2
             Resort Hotel      1378                 23.2
2015-08      City Hotel        2480                 49.7
             Resort Hotel      1409                 26.0
2015-09      City Hotel        3529                 43.7
             Resort Hotel      1585                 34.8
2015-10      City Hotel        3386                 39.0
             Resort Hotel      1571                 26.2
2015-11      City Hotel        1235                 24.4
             Resort Hotel      1105                 16.7
2015-12      City Hotel        1654                 40.4
             Resort Hotel      1266                 24.1
2016-01      City Hotel        1364                 32.1
             Resort Hotel       884                 13.5
2016-02      City Hotel        2371                 39.2
             Resort Hotel      

## 4. Observation: inspect training target and candidate fields

The target and monthly/hotel pattern below use train only. Candidate features are provisional because a Bin 1 judgment is not independent proof of when the source system makes a value available.

In [6]:
print("Training target counts:")
print(train["is_canceled"].value_counts().sort_index().to_string())
print("Training cancellation rate %:", round(train["is_canceled"].mean() * 100, 1))

# Bookings and cancellation rate for each hotel.
hotel_target_summary = train.groupby("hotel")["is_canceled"].agg(
    bookings="count",
    cancellation_rate="mean",
)
hotel_target_summary["cancellation_rate"] = (hotel_target_summary["cancellation_rate"] * 100).round(1)
hotel_target_summary = hotel_target_summary.rename(columns={"cancellation_rate": "cancellation_rate_%"})
print("\nTraining target by hotel:")
print(hotel_target_summary.to_string())

# The four fields you placed in Bin 1.
candidate_columns = [
    "hotel", "is_repeated_guest",
    "previous_cancellations", "previous_bookings_not_canceled",
]
candidate_profile = pd.DataFrame({
    "count": train[candidate_columns].count(),
    "missing": train[candidate_columns].isna().sum(),
    "distinct_values": train[candidate_columns].nunique(),
    "zero_rows": (train[candidate_columns] == 0).sum(),
})
print("\nProvisional candidate fields:")
print(candidate_profile.to_string())

Training target counts:
is_canceled
0    31572
1    17585
Training cancellation rate %: 35.8

Training target by hotel:
              bookings  cancellation_rate_%
hotel                                      
City Hotel       31623                 41.7
Resort Hotel     17534                 25.2

Provisional candidate fields:
                                count  missing  distinct_values  zero_rows
hotel                           49157        0                2          0
is_repeated_guest               49157        0                2      47529
previous_cancellations          49157        0               14      43348
previous_bookings_not_canceled  49157        0               37      47813


## 5. Proposed data-handling decisions - awaiting your approval

The notebook has not removed rows, filled values, or clipped numbers. These are proposals based on the observations, not approved cleaning actions. The detailed branches are below.

### Evidence summary and proposed data actions

Review this table, then answer each decision gate below.

In [7]:
# Missing values in the candidate fields, for each period.
candidate_missing = pd.DataFrame({
    "train": train[candidate_columns].isna().sum(),
    "choose": choose[candidate_columns].isna().sum(),
    "proof": proof[candidate_columns].isna().sum(),
})
print("Missing values in provisional inputs by split:")
print(candidate_missing.to_string())

proposed_data_actions = pd.DataFrame({
    "proposal": [
        "Retain exact repeated rows",
        "Leave missing non-candidate fields unchanged",
        "Do not clip unusual non-negative values",
    ],
    "evidence": [
        "16,291 exact repeated train rows; no booking ID to confirm errors",
        "company, agent, country, children have missing values; candidate fields do not",
        "No negatives in checked train fields; large values alone do not prove errors",
    ],
    "status": "Awaiting approval",
})
print("\nProposals only; no data values changed:")
print(proposed_data_actions.to_string(index=False))

Missing values in provisional inputs by split:
                                train  choose  proof
hotel                               0       0      0
is_repeated_guest                   0       0      0
previous_cancellations              0       0      0
previous_bookings_not_canceled      0       0      0

Proposals only; no data values changed:
                                    proposal                                                                       evidence            status
                  Retain exact repeated rows              16,291 exact repeated train rows; no booking ID to confirm errors Awaiting approval
Leave missing non-candidate fields unchanged company, agent, country, children have missing values; candidate fields do not Awaiting approval
     Do not clip unusual non-negative values   No negatives in checked train fields; large values alone do not prove errors Awaiting approval


## 6. Proposed feature set and engineering - awaiting your approval

> Candidate pool: the four fields you classified as Bin 1. Proposed preparation: one-hot encode `hotel` from train categories and add `log1p` versions of the prior booking counts. Do not include `is_canceled`, `reservation_status`, or `reservation_status_date` as predictors.

The feature matrices below are preview only. They are not final selected inputs and must not be passed to a model until you approve the relevant gates. Point-in-time availability still needs verification.

In [8]:
# Hotel names come from train only: ['City Hotel', 'Resort Hotel'].
hotel_names = sorted(train["hotel"].unique())


def make_features(period):
    """Build the candidate features for one period. The target is not used."""
    features = pd.DataFrame(index=period.index)

    # Raw candidate fields.
    features["is_repeated_guest"] = period["is_repeated_guest"]
    features["previous_cancellations"] = period["previous_cancellations"]
    features["previous_bookings_not_canceled"] = period["previous_bookings_not_canceled"]

    # log1p(x) = log(1 + x), which works for zero counts.
    features["log_previous_cancellations"] = np.log1p(period["previous_cancellations"])
    features["log_previous_bookings_not_canceled"] = np.log1p(period["previous_bookings_not_canceled"])

    # One 0/1 column per hotel, e.g. hotel_city_hotel.
    for hotel_name in hotel_names:
        column_name = "hotel_" + hotel_name.lower().replace(" ", "_")
        features[column_name] = (period["hotel"] == hotel_name).astype(int)

    return features


# Build the same columns for each period separately.
X_train_preview = make_features(train)
X_choose_preview = make_features(choose)
X_proof_preview = make_features(proof)

# Keep the same columns as train, even if a hotel is absent in another period.
X_choose_preview = X_choose_preview.reindex(columns=X_train_preview.columns, fill_value=0)
X_proof_preview = X_proof_preview.reindex(columns=X_train_preview.columns, fill_value=0)

print("Feature columns:")
print(list(X_train_preview.columns))
print("Train feature shape:", X_train_preview.shape)
print("Choose feature shape:", X_choose_preview.shape)
print("Proof feature shape:", X_proof_preview.shape)
print("Missing values in train features:", X_train_preview.isna().sum().sum())
print("Missing values in choose features:", X_choose_preview.isna().sum().sum())
print("Missing values in proof features:", X_proof_preview.isna().sum().sum())

Feature columns:
['is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'log_previous_cancellations', 'log_previous_bookings_not_canceled', 'hotel_city_hotel', 'hotel_resort_hotel']
Train feature shape: (49157, 7)
Choose feature shape: (29546, 7)
Proof feature shape: (40687, 7)
Missing values in train features: 0
Missing values in choose features: 0
Missing values in proof features: 0


## 7. Decision tree: approve or disapprove each proposal

```text
Contract periods → already approved by your direct request → split completed
  ↓ review train-only observations
D1: 16,291 exact repeated train rows, no booking ID
  approve → keep every row | disapprove → pause; specify a reliable deduplication key/rule
D2: missing values are in non-candidate fields; candidate pool has none
  approve → leave missing values as-is | disapprove → pause; specify imputation/recode
D3: unusual values are non-negative; no range evidence proves errors
  approve → retain values | disapprove → pause; specify a supported correction/cap
D4: four learner-Bin-1 fields are candidate inputs, timing not independently verified
  approve → use only as provisional model candidates | disapprove → pause feature selection
D5: prior counts are highly zero-heavy
  approve → add log1p count features | disapprove → use raw counts only
D6: hotel has two training categories
  approve → one-hot encode with train categories | disapprove → pause and choose encoding
  ↓
All approved → create final feature matrices | any disapproved/pending → no final matrices, return to that branch
```

Disapproval never deletes or recodes data automatically; it records where work pauses.

### Decision 1 — retain exact repeated rows?

Observation: 16,291 training rows exactly match an earlier row across the available fields. No reservation ID is available to prove they are invalid duplicates. Proposal: retain all rows.

In [9]:
decision_duplicates = "pending"  # Change to "approve" or "disapprove".
if decision_duplicates == "approve":
    print("Approved: retain all rows; remove none.")
elif decision_duplicates == "disapprove":
    print("Paused: no rows removed. Specify a reliable booking key or deduplication rule first.")
else:
    print("Pending: no duplicate-handling choice has been applied.")

Pending: no duplicate-handling choice has been applied.


### Decision 2 — leave missing non-candidate values unchanged?

Observation: missing values are in `company`, `agent`, `country`, and 4 `children` rows in train. The provisional feature fields have no missing values in any split. Proposal: leave missing values as-is.

In [10]:
decision_missing = "pending"  # Change to "approve" or "disapprove".
if decision_missing == "approve":
    print("Approved: preserve missing values; no imputation needed for current candidate fields.")
elif decision_missing == "disapprove":
    print("Paused: no values filled. Specify which fields and the evidence-based treatment.")
else:
    print("Pending: no missing-value handling choice has been applied.")

Pending: no missing-value handling choice has been applied.


### Decision 3 — retain unusual non-negative values?

Observation: training maxima include 55 adults and ADR 5,400; the checked numeric fields have no negatives. Proposal: retain them unless a source rule confirms an error.

In [11]:
decision_ranges = "pending"  # Change to "approve" or "disapprove".
if decision_ranges == "approve":
    print("Approved: retain unusual non-negative values; no clipping applied.")
elif decision_ranges == "disapprove":
    print("Paused: no values clipped. Specify a supported validity rule first.")
else:
    print("Pending: no range-handling choice has been applied.")

Pending: no range-handling choice has been applied.


### Decision 4 — use the four Bin 1 fields as provisional candidates?

Observation: `hotel`, `is_repeated_guest`, `previous_cancellations`, and `previous_bookings_not_canceled` have no missing values. Their exact availability at the intended prediction time still needs confirmation. Proposal: allow them into model comparisons provisionally, not as production-approved inputs.

In [12]:
decision_features = "pending"  # Change to "approve" or "disapprove".
if decision_features == "approve":
    print("Approved provisionally: use the four audited fields for offline comparisons; timing caveat remains.")
elif decision_features == "disapprove":
    print("Paused: do not use these predictors until an alternative timing-safe set is agreed.")
else:
    print("Pending: no predictor set has final approval.")

Pending: no predictor set has final approval.


### Decision 5 — add log1p versions of the prior counts?

Observation: 43,348 of 49,157 training rows have zero prior cancellations; 47,813 have zero prior non-cancelled bookings. Proposal: add `log1p` versions alongside the raw counts.

In [13]:
decision_log_features = "pending"  # Change to "approve" or "disapprove".
if decision_log_features == "approve":
    print("Approved: include both log1p prior-count features in the candidate matrix.")
elif decision_log_features == "disapprove":
    print("Disapproved: use the raw prior counts only; rebuild the preview without log1p fields.")
else:
    print("Pending: log1p features remain a preview only.")

Pending: log1p features remain a preview only.


### Decision 6 — one-hot encode hotel?

Observation: train has two hotel categories, both present in choose and proof. Proposal: create two indicator columns using categories observed in train.

In [14]:
decision_hotel_encoding = "pending"  # Change to "approve" or "disapprove".
if decision_hotel_encoding == "approve":
    print("Approved: use the two train-derived hotel indicator columns.")
elif decision_hotel_encoding == "disapprove":
    print("Paused: choose an alternative encoding before creating final matrices.")
else:
    print("Pending: hotel encoding remains a preview only.")

Pending: hotel encoding remains a preview only.


### Final gate — build approved feature matrices only

Run after editing all six decision cells. If any choice is pending or disapproved, this cell will not create final feature matrices.

In [15]:
decisions = pd.Series({
    "duplicates": decision_duplicates,
    "missing_values": decision_missing,
    "numeric_ranges": decision_ranges,
    "candidate_features": decision_features,
    "log_features": decision_log_features,
    "hotel_encoding": decision_hotel_encoding,
})
print("Current decisions:")
print(decisions.to_string())

# log_features may be "approve" or "disapprove". Every other gate must be "approve".
other_decisions = decisions.drop("log_features")
not_approved = other_decisions[other_decisions != "approve"]
log_choice_made = decision_log_features in ["approve", "disapprove"]

if len(not_approved) > 0 or not log_choice_made:
    print("\nFinal feature matrices not created.")
    print("Need approval:", list(not_approved.index))
    if not log_choice_made:
        print("log_features must be set to approve or disapprove.")
else:
    X_train = X_train_preview.copy()
    X_choose = X_choose_preview.copy()
    X_proof = X_proof_preview.copy()

    if decision_log_features == "disapprove":
        log_columns = ["log_previous_cancellations", "log_previous_bookings_not_canceled"]
        X_train = X_train.drop(columns=log_columns)
        X_choose = X_choose.drop(columns=log_columns)
        X_proof = X_proof.drop(columns=log_columns)

    print("\nRequired data-handling and feature choices approved.")
    print("Log1p features:", decision_log_features)
    print("Final feature matrix shapes:", X_train.shape, X_choose.shape, X_proof.shape)

Current decisions:
duplicates            pending
missing_values        pending
numeric_ranges        pending
candidate_features    pending
log_features          pending
hotel_encoding        pending

Final feature matrices not created.
Need approval: ['duplicates', 'missing_values', 'numeric_ranges', 'candidate_features', 'hotel_encoding']
log_features must be set to approve or disapprove.


## Current result and decision status

- Data loaded and split by the requested contract periods.
- Training-period EDA and cleaning proposals are visible above. The raw data has not been changed.
- Candidate feature matrices are previews only.
- No model has been trained and no proof cancellation outcome has been inspected.

The final approval summary below will create `X_train`, `X_choose`, and `X_proof` only after every required gate is approved. If any gate is disapproved or pending, it stops and reports which decision needs discussion.